# 빅데이터 처리와 시각화 8장 - Pandas 데이터 분석 심화
## 실무 데이터 분석의 핵심 패턴

이번 장에서는 **실제 현업에서 빈번하게 사용하는 분석 패턴**을 집중적으로 학습.

<div class="alert alert-block alert-info">
<b>💡 이번 장에서 배우는 것</b><br>
7장까지의 데이터 정제 기술을 기반으로, 정제된 데이터로 실질적인 비즈니스 인사이트를 도출하는 분석 기법을 익힘.
</div>

| 섹션 | 주제 | 핵심 함수 |
| :--- | :--- | :--- |
| 1 | 피벗 테이블 | `pivot_table()` |
| 2 | 데이터 구간화 | `pd.cut()`, `pd.qcut()` |
| 3 | 시계열 분석 심화 | `rolling()`, `resample()` |

In [2]:
import pandas as pd
import numpy as np

---
## 1. 피벗 테이블 (Pivot Table)
- **피벗 테이블**: 데이터를 행/열 기준으로 교차 집계하여 한눈에 보기 쉬운 요약표를 만드는 기법.
- 엑셀의 '피벗 테이블' 기능과 동일한 개념이며, Pandas에서는 `pivot_table()`로 구현.
- `groupby().agg()`보다 **다차원 요약**에 특화되어 있음.

<div class="alert alert-block alert-info">
<b>💡 언제 쓰나?</b> "부서별 × 직급별 평균 월급", "지역별 × 카테고리별 총 매출" 처럼 두 가지 기준으로 교차 분석할 때.
</div>

#### 1-1. pivot_table() 기본 사용법

```python
df.pivot_table(
    values='집계할 컬럼',
    index='행 기준 컬럼',
    columns='열 기준 컬럼',
    aggfunc='집계 함수',   # 기본값: 'mean'
    fill_value=0,          # NaN을 대체할 값
    margins=True           # 합계 행/열 추가
)
```

| 파라미터 | 설명 | 예시 |
| :--- | :--- | :--- |
| `values` | 집계 대상 컬럼 | `'매출'` |
| `index` | 행(Row) 기준 | `'지역'` |
| `columns` | 열(Column) 기준 | `'카테고리'` |
| `aggfunc` | 집계 방법 | `'sum'`, `'mean'`, `'count'` |
| `margins` | All(합계) 행/열 추가 | `True` |
| `fill_value` | 빈 셀 채울 값 | `0` |

In [5]:
# 피벗 테이블 실습 데이터 - 쇼핑몰 판매 기록
np.random.seed(42)

data_sales = {
    '주문ID':   [f'ORD-{i:04d}' for i in range(1, 25)],
    '지역':     ['서울', '서울', '부산', '부산', '대구', '서울', '부산', '대구',
                 '서울', '서울', '부산', '서울', '대구', '부산', '대구', '서울',
                 '부산', '서울', '대구', '서울', '부산', '대구', '서울', '부산'],
    '카테고리':  ['전자제품', '의류', '전자제품', '식품', '의류', '식품', '전자제품', '의류',
                 '식품', '전자제품', '의류', '전자제품', '식품', '전자제품', '의류', '식품',
                 '전자제품', '의류', '전자제품', '식품', '의류', '전자제품', '식품', '의류'],
    '분기':     ['Q1', 'Q1', 'Q1', 'Q1', 'Q1', 'Q1', 'Q2', 'Q2',
                 'Q2', 'Q2', 'Q2', 'Q3', 'Q3', 'Q3', 'Q3', 'Q3',
                 'Q3', 'Q4', 'Q4', 'Q4', 'Q4', 'Q4', 'Q4', 'Q4'],
    '매출':     [1200000, 85000, 950000, 42000, 120000, 55000, 870000, 95000,
                 38000, 1100000, 76000, 980000, 61000, 1050000, 88000, 49000,
                 920000, 105000, 1300000, 72000, 91000, 1150000, 66000, 83000],
    '수량':     [2, 3, 1, 5, 2, 4, 1, 3, 6, 2, 4, 1, 7, 2, 3, 5, 1, 2, 3, 8, 4, 2, 9, 3]
}

df_sales = pd.DataFrame(data_sales)
print(f'데이터 크기: {df_sales.shape}')
display(df_sales.head(8))

데이터 크기: (24, 6)


,주문ID,지역,카테고리,분기,매출,수량
0,ORD-0001,서울,전자제품,Q1,1200000,2
1,ORD-0002,서울,의류,Q1,85000,3
2,ORD-0003,부산,전자제품,Q1,950000,1
3,ORD-0004,부산,식품,Q1,42000,5
4,ORD-0005,대구,의류,Q1,120000,2
5,ORD-0006,서울,식품,Q1,55000,4
6,ORD-0007,부산,전자제품,Q2,870000,1
7,ORD-0008,대구,의류,Q2,95000,3


In [6]:
# 기본 피벗 테이블: 지역 × 카테고리별 총 매출
pivot_basic = df_sales.pivot_table(
    values='매출',
    index='지역',
    columns='카테고리',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='전체합계'
)

print('=== 지역 × 카테고리별 총 매출 ===')
display(pivot_basic)

=== 지역 × 카테고리별 총 매출 ===


카테고리,식품,의류,전자제품,전체합계
지역,,,,
대구,61000,303000,2450000,2814000
부산,42000,250000,3790000,4082000
서울,280000,190000,3280000,3750000
전체합계,383000,743000,9520000,10646000


In [7]:
# 다양한 집계 함수 적용
# 1. 건수(count) 피벗
print('=== 지역 × 카테고리별 주문 건수 ===')
pivot_count = df_sales.pivot_table(
    values='주문ID',
    index='지역',
    columns='카테고리',
    aggfunc='count',
    fill_value=0
)
display(pivot_count)

# 2. 평균 매출 피벗
print('\n=== 분기 × 카테고리별 평균 매출 ===')
pivot_avg = df_sales.pivot_table(
    values='매출',
    index='분기',
    columns='카테고리',
    aggfunc='mean',
    fill_value=0
)
display(pivot_avg)

=== 지역 × 카테고리별 주문 건수 ===


카테고리,식품,의류,전자제품
지역,,,
대구,1,3,2
부산,1,3,4
서울,5,2,3



=== 분기 × 카테고리별 평균 매출 ===


카테고리,식품,의류,전자제품
분기,,,
Q1,48500.0,102500.0,1.075000e+06
Q2,38000.0,85500.0,9.850000e+05
Q3,55000.0,88000.0,9.833333e+05
Q4,69000.0,93000.0,1.225000e+06


#### 1-2. 피벗 테이블 결과 활용

- 피벗 테이블 결과는 일반 DataFrame이므로 인덱싱, 정렬, 추가 연산 모두 가능.
- **`reset_index()`**: 인덱스를 일반 컬럼으로 변환하여 재사용에 용이하게 만들 수 있음.

In [9]:
# 피벗 테이블을 다시 활용
pivot_region = df_sales.pivot_table(
    values='매출',
    index='지역',
    columns='카테고리',
    aggfunc='sum',
    fill_value=0
)

# 1. 특정 컬럼 조회
print('=== 지역별 전자제품 매출 ===')
print(pivot_region['전자제품'])

# 2. 비율 계산: 각 지역에서 카테고리별 매출 비중
print('\n=== 지역별 카테고리 매출 비중(%) ===')
pivot_pct = pivot_region.div(pivot_region.sum(axis=1), axis=0) * 100
display(pivot_pct.round(1))

# 3. reset_index() 로 일반 DataFrame 변환
print('\n=== reset_index() 후 ===')
pivot_flat = pivot_region.reset_index()
display(pivot_flat)

=== 지역별 전자제품 매출 ===
지역
대구    2450000
부산    3790000
서울    3280000
Name: 전자제품, dtype: int64

=== 지역별 카테고리 매출 비중(%) ===


카테고리,식품,의류,전자제품
지역,,,
대구,2.2,10.8,87.1
부산,1.0,6.1,92.8
서울,7.5,5.1,87.5



=== reset_index() 후 ===


카테고리,지역,식품,의류,전자제품
0,대구,61000,303000,2450000
1,부산,42000,250000,3790000
2,서울,280000,190000,3280000


> #### Exercise
> 위의 `df_sales` 데이터를 사용하여 다음을 수행하시오.
> 1. `분기`를 index, `지역`을 columns로 하는 **총 수량 피벗 테이블**을 만드시오.
> 2. 위 피벗 테이블에서 `margins=True`를 추가하여 분기별/지역별 합계도 함께 출력하시오.
> 3. `지역 × 카테고리` 조합별 **총 매출 상위 3개**를 찾으시오.
> (힌트: pivot_table 결과를 `stack()`으로 변환 후 정렬)

In [11]:
# 여기에 코드를 작성하시오.


> #### Advance
> - `df_sales`에서 `aggfunc` 자리에 리스트를 넣어 **sum과 count를 동시에** 집계하는 피벗 테이블을 만드시오.
> - 결과에서 전자제품 매출 합계와 전자제품 주문 건수를 이용해 **전자제품 평균 단가(매출합계 / 건수)** 를 지역별로 계산하시오.
> 
> ```python
> # 힌트
> pivot_multi = df_sales.pivot_table(
>     values='매출',
>     index='지역',
>     columns='카테고리',
>     aggfunc=['sum', 'count']
> )
> ```

In [13]:
# 여기에 코드를 작성하시오.


---
## 2. 데이터 구간화 (Binning)
- **구간화(Binning)**: 연속형 수치 데이터를 특정 기준으로 구간(범주)으로 나누는 작업.
- 실무에서 '연령대', '등급', '성과 구간' 같은 카테고리를 만들 때 필수적으로 사용.

| 함수 | 기준 | 사용 상황 |
| :--- | :--- | :--- |
| `pd.cut()` | **직접 지정한 범위** | "20~29세는 20대" 처럼 구간 폭을 내가 정할 때 |
| `pd.qcut()` | **동일한 데이터 개수** | "상위 25%, 중상위 25%..." 처럼 분위수로 나눌 때 |

#### 2-1. 직접 구간 지정: `pd.cut()`

```python
pd.cut(
    x,              # 구간화할 Series
    bins,           # 구간 경계값 리스트 또는 구간 개수
    labels=None,    # 각 구간에 붙일 이름
    right=True,     # 오른쪽 경계 포함 여부 (기본: 포함)
    include_lowest=True  # 가장 낮은 값 포함
)
```

In [16]:
# 구간화 실습 데이터 - 고객 정보
data_customer = {
    '고객ID':    [f'C{i:03d}' for i in range(1, 16)],
    '나이':      [23, 35, 42, 28, 51, 19, 67, 31, 44, 56, 38, 25, 72, 48, 33],
    '연간구매액': [850000, 2300000, 1500000, 450000, 3800000, 120000, 5200000,
                  1800000, 2700000, 4100000, 950000, 330000, 6800000, 3200000, 1100000],
    '구매횟수':  [3, 12, 7, 2, 18, 1, 24, 9, 14, 20, 4, 2, 30, 15, 5]
}

df_cust = pd.DataFrame(data_customer)
display(df_cust)

,고객ID,나이,연간구매액,구매횟수
0,C001,23,850000,3
1,C002,35,2300000,12
2,C003,42,1500000,7
3,C004,28,450000,2
4,C005,51,3800000,18
5,C006,19,120000,1
6,C007,67,5200000,24
7,C008,31,1800000,9
8,C009,44,2700000,14
9,C010,56,4100000,20


In [17]:
# pd.cut() - 직접 구간 지정
# 나이를 연령대로 구분
bins_age = [0, 29, 39, 49, 59, 100]
labels_age = ['20대 이하', '30대', '40대', '50대', '60대 이상']

df_cust['연령대'] = pd.cut(
    df_cust['나이'],
    bins=bins_age,
    labels=labels_age,
    right=True
)

print('=== 연령대 구간화 결과 ===')
display(df_cust[['고객ID', '나이', '연령대']].sort_values('나이'))

print('\n=== 연령대별 고객 수 ===')
print(df_cust['연령대'].value_counts().sort_index())

=== 연령대 구간화 결과 ===


,고객ID,나이,연령대
5,C006,19,20대 이하
0,C001,23,20대 이하
11,C012,25,20대 이하
3,C004,28,20대 이하
7,C008,31,30대
14,C015,33,30대
1,C002,35,30대
10,C011,38,30대
2,C003,42,40대
8,C009,44,40대



=== 연령대별 고객 수 ===
연령대
20대 이하    4
30대       4
40대       3
50대       2
60대 이상    2
Name: count, dtype: int64


In [18]:
# 연간구매액으로 고객 등급 나누기
bins_grade = [0, 500000, 1500000, 3000000, float('inf')]
labels_grade = ['일반', '실버', '골드', 'VIP']

df_cust['고객등급'] = pd.cut(
    df_cust['연간구매액'],
    bins=bins_grade,
    labels=labels_grade,
    right=True
)

print('=== 고객 등급 분류 결과 ===')
display(df_cust[['고객ID', '나이', '연간구매액', '고객등급']].sort_values('연간구매액'))

print('\n=== 등급별 통계 ===')
display(df_cust.groupby('고객등급', observed=True)['연간구매액'].agg(['count', 'mean', 'sum']))

=== 고객 등급 분류 결과 ===


,고객ID,나이,연간구매액,고객등급
5,C006,19,120000,일반
11,C012,25,330000,일반
3,C004,28,450000,일반
0,C001,23,850000,실버
10,C011,38,950000,실버
14,C015,33,1100000,실버
2,C003,42,1500000,실버
7,C008,31,1800000,골드
1,C002,35,2300000,골드
8,C009,44,2700000,골드



=== 등급별 통계 ===


,count,mean,sum
고객등급,,,
일반,3,3.000000e+05,900000
실버,4,1.100000e+06,4400000
골드,3,2.266667e+06,6800000
VIP,5,4.620000e+06,23100000


#### 2-2. 분위수 기반 구간화: `pd.qcut()`

- `pd.cut()`은 **구간의 폭**을 기준으로 나누지만, `pd.qcut()`은 **각 구간에 들어가는 데이터 수**가 동일하게 나뉨.
- 상위/하위 몇 %와 같은 **상대적 순위** 기반 분류에 적합.

```python
pd.qcut(
    x,
    q,        # 분위수 개수 (4이면 4등분 → 25%, 50%, 75%, 100%)
    labels=None
)
```

In [20]:
# pd.qcut() - 분위수 기반 구간화
# 구매액을 4분위로 나눔 (각 구간에 동일한 수의 고객)
df_cust['구매액_분위'] = pd.qcut(
    df_cust['연간구매액'],
    q=4,
    labels=['하위25%', '중하위', '중상위', '상위25%']
)

print('=== qcut: 분위수 기반 구간 (각 구간 고객 수가 동일) ===')
display(df_cust[['고객ID', '연간구매액', '고객등급', '구매액_분위']].sort_values('연간구매액'))

print('\n--- pd.cut vs pd.qcut 비교 ---')
compare = df_cust.groupby('구매액_분위', observed=True)['연간구매액'].agg(['count', 'min', 'max'])
compare.columns = ['고객수', '최솟값', '최댓값']
display(compare)

=== qcut: 분위수 기반 구간 (각 구간 고객 수가 동일) ===


,고객ID,연간구매액,고객등급,구매액_분위
5,C006,120000,일반,하위25%
11,C012,330000,일반,하위25%
3,C004,450000,일반,하위25%
0,C001,850000,실버,하위25%
10,C011,950000,실버,중하위
14,C015,1100000,실버,중하위
2,C003,1500000,실버,중하위
7,C008,1800000,골드,중하위
1,C002,2300000,골드,중상위
8,C009,2700000,골드,중상위



--- pd.cut vs pd.qcut 비교 ---


,고객수,최솟값,최댓값
구매액_분위,,,
하위25%,4,120000,850000
중하위,4,950000,1800000
중상위,3,2300000,3200000
상위25%,4,3800000,6800000


> #### Exercise
> 위의 `df_cust` 데이터를 사용하여 다음을 수행하시오.
> 1. `구매횟수`를 기준으로 `pd.cut()`을 사용하여 다음 기준의 `구매빈도등급` 컬럼을 추가하시오.
>    - 0-5회: '저빈도', 6-14회: '중빈도', 15회 이상: '고빈도'
> 2. `고객등급`과 `구매빈도등급`을 기준으로 groupby하여 평균 연간구매액을 구하시오.
> 3. `연간구매액`을 5분위(`q=5`)로 나누어 `구매액_5분위` 컬럼을 추가하고 각 분위별 고객 수를 출력하시오.

In [22]:
# 여기에 코드를 작성하시오.


---
## 3. 시계열 분석 심화
- 5장에서 배운 `pd.to_datetime()`, `dt accessor`를 넘어, **시간 흐름에 따른 패턴**을 분석하는 기법.
- 주가, 매출, 날씨, 트래픽 등 **날짜/시간이 핵심인 데이터**를 다룰 때 필수.

| 기법 | 함수 | 설명 |
| :--- | :--- | :--- |
| 이동 평균 | `rolling()` | 일정 기간의 평균을 슬라이딩 윈도우로 계산 |
| 누적 집계 | `expanding()` | 처음부터 현재까지 누적 계산 |
| 기간 재집계 | `resample()` | 일별 → 주별/월별 등 집계 단위 변경 |

#### 3-1. 이동 평균: `rolling()`

- **이동 평균(Moving Average)**: 최근 N개 데이터의 평균값. 노이즈를 제거하고 추세를 파악하는 데 사용.
- 주가 분석에서 '5일 이동평균선', '20일 이동평균선'이 대표적인 예.

```python
df['컬럼'].rolling(window=N).mean()   # N개 기간 이동 평균
df['컬럼'].rolling(window=N).sum()    # N개 기간 이동 합계
df['컬럼'].rolling(window=N).std()    # N개 기간 이동 표준편차
```

In [25]:
# 시계열 실습 데이터 - 일별 온라인 쇼핑몰 매출 (3개월)
np.random.seed(0)
dates = pd.date_range(start='2024-01-01', end='2024-03-31', freq='D')

# 기본 트렌드 + 요일 효과 + 무작위 노이즈
trend = np.linspace(500000, 800000, len(dates))          # 상승 트렌드
weekday_effect = np.where(pd.Series(dates).dt.dayofweek >= 5, 1.3, 1.0)  # 주말 30% 증가
noise = np.random.normal(0, 50000, len(dates))           # 랜덤 노이즈

df_ts = pd.DataFrame({
    '날짜': dates,
    '일별매출': (trend * weekday_effect + noise).astype(int),
    '방문자수': (np.random.normal(1500, 200, len(dates))).astype(int)
})
df_ts = df_ts.set_index('날짜')  # 날짜를 인덱스로 설정

print(f'데이터 기간: {df_ts.index[0].date()} ~ {df_ts.index[-1].date()}')
print(f'총 {len(df_ts)}일 데이터')
display(df_ts.head(10))

데이터 기간: 2024-01-01 ~ 2024-03-31
총 91일 데이터


,일별매출,방문자수
날짜,,
2024-01-01,588202,1744
2024-01-02,523341,1541
2024-01-03,555603,1695
2024-01-04,622044,1571
2024-01-05,606711,1641
2024-01-06,622802,1502
2024-01-07,723504,1857
2024-01-08,515765,1525
2024-01-09,521505,1580


In [26]:
# 이동 평균 계산
df_ts['MA_7일']  = df_ts['일별매출'].rolling(window=7).mean().round(0)   # 7일 이동평균
df_ts['MA_14일'] = df_ts['일별매출'].rolling(window=14).mean().round(0)  # 14일 이동평균

print('=== 이동 평균 결과 (처음 20일) ===')
print('* 7일 이동평균은 7일째부터, 14일은 14일째부터 계산됨 (이전은 NaN)')
display(df_ts[['일별매출', 'MA_7일', 'MA_14일']].head(20))

# 이동 표준편차 - 변동성 파악
df_ts['변동성_7일'] = df_ts['일별매출'].rolling(window=7).std().round(0)
print('\n=== 7일 이동 표준편차 (변동성) - 마지막 10일 ===')
display(df_ts[['일별매출', 'MA_7일', '변동성_7일']].tail(10))

=== 이동 평균 결과 (처음 20일) ===
* 7일 이동평균은 7일째부터, 14일은 14일째부터 계산됨 (이전은 NaN)


,일별매출,MA_7일,MA_14일
날짜,,,
2024-01-01,588202,NaN,NaN
2024-01-02,523341,NaN,NaN
2024-01-03,555603,NaN,NaN
2024-01-04,622044,NaN,NaN
2024-01-05,606711,NaN,NaN
2024-01-06,622802,NaN,NaN
2024-01-07,723504,606030.0,NaN
2024-01-08,515765,595681.0,NaN
2024-01-09,521505,595419.0,NaN



=== 7일 이동 표준편차 (변동성) - 마지막 10일 ===


,일별매출,MA_7일,변동성_7일
날짜,,,
2024-03-22,815041,805470.0,96711.0
2024-03-23,1028616,818021.0,120074.0
2024-03-24,932854,817601.0,119597.0
2024-03-25,854412,835699.0,113141.0
2024-03-26,878127,854800.0,106239.0
2024-03-27,845605,866152.0,99199.0
2024-03-28,781003,876523.0,82317.0
2024-03-29,739795,865773.0,95536.0
2024-03-30,1088389,874312.0,113499.0


#### 3-2. 누적 집계: `expanding()`

- **`expanding()`**: 처음 데이터부터 현재까지 **누적**하여 계산.
- `rolling()`은 고정된 윈도우 크기, `expanding()`은 윈도우가 계속 커짐.
- 누적 매출, 누적 최고/최저 기록 추적에 활용.

In [28]:
# 누적 통계 계산
df_ts['누적매출']   = df_ts['일별매출'].cumsum()           # .expanding().sum()과 동일
df_ts['누적최고매출'] = df_ts['일별매출'].expanding().max() # 지금까지의 최고 매출
df_ts['누적평균매출'] = df_ts['일별매출'].expanding().mean().round(0)

print('=== 누적 통계 (처음 10일) ===')
display(df_ts[['일별매출', '누적매출', '누적최고매출', '누적평균매출']].head(10))

# 오늘 매출이 누적 평균 대비 얼마나 좋은지 확인
df_ts['평균대비(%)'] = (df_ts['일별매출'] / df_ts['누적평균매출'] * 100 - 100).round(1)
print('\n=== 일별 매출의 누적 평균 대비 성과 (마지막 10일) ===')
display(df_ts[['일별매출', '누적평균매출', '평균대비(%)']].tail(10))

=== 누적 통계 (처음 10일) ===


,일별매출,누적매출,누적최고매출,누적평균매출
날짜,,,,
2024-01-01,588202,588202,588202.0,588202.0
2024-01-02,523341,1111543,588202.0,555772.0
2024-01-03,555603,1667146,588202.0,555715.0
2024-01-04,622044,2289190,622044.0,572298.0
2024-01-05,606711,2895901,622044.0,579180.0
2024-01-06,622802,3518703,622802.0,586450.0
2024-01-07,723504,4242207,723504.0,606030.0
2024-01-08,515765,4757972,723504.0,594746.0
2024-01-09,521505,5279477,723504.0,586609.0



=== 일별 매출의 누적 평균 대비 성과 (마지막 10일) ===


,일별매출,누적평균매출,평균대비(%)
날짜,,,
2024-03-22,815041,684458.0,19.1
2024-03-23,1028616,688605.0,49.4
2024-03-24,932854,691512.0,34.9
2024-03-25,854412,693429.0,23.2
2024-03-26,878127,695577.0,26.2
2024-03-27,845605,697301.0,21.3
2024-03-28,781003,698252.0,11.9
2024-03-29,739795,698719.0,5.9
2024-03-30,1088389,703049.0,54.8


#### 3-3. 기간 재집계: `resample()`

- **`resample()`**: 시계열 데이터의 **집계 단위를 변경**하는 함수.
- 일별 → 주별, 일별 → 월별 등으로 집계할 때 사용.
- `resample()`은 **DatetimeIndex**가 설정된 DataFrame에서만 작동.

| 주요 주기 코드 | 설명 |
| :--- | :--- |
| `'D'` | 일별 (Day) |
| `'W'` | 주별 (Week, 일요일 기준) |
| `'W-MON'` | 주별 (월요일 기준) |
| `'ME'` | 월별 마지막 날 (Month End) |
| `'QE'` | 분기별 마지막 날 (Quarter End) |

In [30]:
# resample() - 일별 데이터를 주별/월별로 재집계
# 1. 주별 집계
weekly = df_ts.resample('W-MON')['일별매출'].agg([
    ('주간총매출', 'sum'),
    ('주간평균매출', 'mean'),
    ('최대일매출', 'max'),
    ('영업일수', 'count')
]).round(0)

print('=== 주별 매출 집계 ===')
display(weekly)

# 2. 월별 집계
monthly = df_ts[['일별매출', '방문자수']].resample('ME').agg({
    '일별매출': ['sum', 'mean', 'max'],
    '방문자수': ['sum', 'mean']
}).round(0)

print('\n=== 월별 매출 및 방문자 집계 ===')
display(monthly)

=== 주별 매출 집계 ===


,주간총매출,주간평균매출,최대일매출,영업일수
날짜,,,,
2024-01-01,588202,588202.0,588202,1
2024-01-08,4169770,595681.0,723504,7
2024-01-15,4243276,606182.0,740051,7
2024-01-22,4218106,602587.0,689628,7
2024-01-29,4552781,650397.0,764954,7
2024-02-05,4560780,651540.0,779937,7
2024-02-12,4669406,667058.0,770905,7
2024-02-19,4903387,700484.0,892541,7
2024-02-26,5058997,722714.0,909749,7



=== 월별 매출 및 방문자 집계 ===


일별매출                      방문자수        
                 sum      mean      max    sum    mean
날짜                                                    
2024-01-31  19050016  614517.0   764954  49487  1596.0
2024-02-29  19959958  688274.0   909749  44196  1524.0
2024-03-31  25284244  815621.0  1088389  45750  1476.0

In [31]:
# 전주 대비 성장률 계산 (pct_change)
weekly_sales = df_ts['일별매출'].resample('W-MON').sum()
weekly_growth = weekly_sales.pct_change() * 100  # 전주 대비 % 변화

result_weekly = pd.DataFrame({
    '주간매출': weekly_sales,
    '전주대비성장률(%)': weekly_growth.round(1)
})

print('=== 주간 매출 및 전주 대비 성장률 ===')
display(result_weekly)

=== 주간 매출 및 전주 대비 성장률 ===


,주간매출,전주대비성장률(%)
날짜,,
2024-01-01,588202,NaN
2024-01-08,4169770,608.9
2024-01-15,4243276,1.8
2024-01-22,4218106,-0.6
2024-01-29,4552781,7.9
2024-02-05,4560780,0.2
2024-02-12,4669406,2.4
2024-02-19,4903387,5.0
2024-02-26,5058997,3.2


> #### Exercise
> 위의 `df_ts` (일별 매출/방문자 데이터)를 사용하여 다음을 수행하시오.
> 1. `방문자수`의 **3일 이동 평균**과 **7일 이동 평균**을 각각 컬럼으로 추가하시오.
> 2. 일별 매출 데이터에서 **`pct_change()`**를 사용하여 전일 대비 매출 변화율(%)을 계산하고, 매출이 전일 대비 10% 이상 상승한 날짜를 출력하시오.
> 3. `resample('W-MON')`을 사용하여 **주별 방문자 합계**를 구하고, **방문자당 매출**(주간총매출 / 주간방문자수)을 계산하시오.

In [33]:
# 여기에 코드를 작성하시오.


> #### Advance - 전년 동기 대비 분석 (YoY)
> - 실무에서 가장 많이 쓰는 비교 지표 중 하나가 **전년 동기 대비(YoY, Year-over-Year)** 성장률.
> - 아래 2년치 데이터에서 월별 매출을 집계하고, `shift(12)`를 활용해 전년 동월 대비 성장률을 계산하시오.

In [35]:

# 2년치 데이터 생성
np.random.seed(1)
dates2 = pd.date_range('2023-01-01', '2024-12-31', freq='D')
df_2y = pd.DataFrame({
     '날짜': dates2,
     '일별매출': (np.linspace(400000, 900000, len(dates2)) + np.random.normal(0, 60000, len(dates2))).astype(int)
}).set_index('날짜')

# 여기에 코드를 작성하시오.
df_2y

,일별매출
날짜,
2023-01-01,497460
2023-01-02,363979
2023-01-03,369679
2023-01-04,337676
2023-01-05,454664
...,...
2024-12-27,903742
2024-12-28,899710
2024-12-29,890542


---
## 8장 핵심 정리

| 기능 | 핵심 함수 | 한 줄 요약 |
| :--- | :--- | :--- |
| 피벗 테이블 | `pivot_table()` | 행/열 기준의 교차 집계표 생성 |
| 직접 구간화 | `pd.cut()` | 내가 정한 경계값으로 구간 나누기 |
| 분위수 구간화 | `pd.qcut()` | 데이터 수가 동일하도록 구간 나누기 |
| 다중 조건 처리 | `np.select()` | 3개 이상 조건의 분기 처리 |
| 이동 평균 | `rolling().mean()` | 최근 N기간 추세 파악 |
| 누적 집계 | `expanding()` / `cumsum()` | 처음부터 현재까지 누적 값 |
| 기간 재집계 | `resample()` | 일별 → 주별/월별 단위 변환 |
| 전기 대비 성장률 | `pct_change()` | 전일/전월 대비 변화율 계산 |

<div class="alert alert-block alert-success">
<b>실무 분석 파이프라인 요약</b><br>
① 데이터 로드 → ② 탐색 및 품질 확인 (결측치, 타입) → ③ 정제 → ④ 파생 변수 생성 (구간화, 조건부 컬럼) → ⑤ 집계 분석 (groupby, pivot_table) → ⑥ 시계열 분석 (rolling, resample) → ⑦ 인사이트 도출 및 보고
</div>